# MLP Model Developmnt - NIH Chest X-ray Classification

### Objective

This notebook develops a simple Multilater Perceptron (MLP) using PyTorch to classify chest X-ray images into two categories: Effusion and No Finding.

The purpose is to understand the fundaments of deep learning and establish a baseline imaging model before exploring Convolutional Neural Netowkrs (CNNs).

### Execution Plan

In this notebook, we will
1. Load the preprocessed chest X-ray images and labels
2. Separate training, validation, and test datasets
3. Create PyTorch Datasets and DataLoaders
4. Build the MLP architecture
5. Implement the training loop and monitor model performance
6. Evaluate the trained model

### Learning Focus

Understand how tensors, neural network layers, loss functions, optimizers, backpropagation, and weight updates work together during model training.

### Limitations

This is an educational project and is not intended for clinical diagnosis.

## 01. Load data and set up environment

In [5]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "processed"

# Load preprocessed tensors
data = torch.load(
    DATA_DIR / "mlp_images_64x64.pt",
    map_location="cpu",
    weights_only=False)
X = data["X"]
y = data["y"]
image_names = data["image_names"]
patient_ids = data["patient_ids"]
splits = data["splits"]

print("Images:", X.shape)
print("Labels:", y.shape)
print("Number of image names:", len(image_names))
print("Number of patient IDs:", len(patient_ids))
print("Number of split assignments:", len(splits))

Images: torch.Size([3241, 1, 64, 64])
Labels: torch.Size([3241])
Number of image names: 3241
Number of patient IDs: 3241
Number of split assignments: 3241


## 02. Prepare the datasets

In [6]:
# Convert split information into a tensor
split_labels = torch.tensor([
    {"train": 0, "val": 1, "test": 2}[s]
    for s in splits])

# Select images for each split
train_mask = split_labels == 0
val_mask = split_labels == 1
test_mask = split_labels == 2

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: torch.Size([2208, 1, 64, 64]) torch.Size([2208])
Validation: torch.Size([554, 1, 64, 64]) torch.Size([554])
Testing: torch.Size([479, 1, 64, 64]) torch.Size([479])


## 03. Build the MLP

### 3.1 Creating PyTorch Datasets and DataLoaders

Datasets store and provide access to individual sample and their labels.
DataLoader retrieves samples from the Dataset and groups them into batches. Training will be performed on these manageable subsets of data. For this, we will be using mini-batch gradient descent with a batch size of 32.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create datasets. TensorDataset combines two tensors (X_train images and y_train corresponding labels) and associates each image with its correct label.
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Training samples: 2208
Validation samples: 554
Test samples: 479


In [9]:
# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True)       #Randomise sample order during training
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False)      #Keep evaluation predictable
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False)      #Keep final eval predictable
    
# Inspect 1 batch
images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print(f"First 10 labels: {labels[:10]}")

Batch image shape: torch.Size([32, 1, 64, 64])
Batch label shape: torch.Size([32])
First 10 labels: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


There are 32 images in this batch, with 1 grayscale channel, and image heights and widths of 64x64. The neural network will receive the images in this format.

### 3.2 Building the MLP Architecture

1. First linear layer (nn.Linear(4096, 128))
2. ReLU activation
3. Second linear layer (nn.Linear(128, 2))

In [10]:
class ChestXrayMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64, 128),
            nn.ReLU(),
            nn.Linear(128, 2))
    def forward(self, x):
        return self.network(x)

model = ChestXrayMLP()
print(model)

ChestXrayMLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [13]:
# Test model with 1 batch
images, labels = next(iter(train_loader))
outputs = model(images)
print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print(f"First 5 predictions (logits): {outputs[:5]}")

# Count model parameters - how many individual weights and biases model can learn
total_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad)

print("Trainable parameters:", total_params)

Input shape: torch.Size([32, 1, 64, 64])
Output shape: torch.Size([32, 2])
First 5 predictions (logits): tensor([[-0.1196,  0.1863],
        [-0.1738,  0.2653],
        [-0.2108,  0.2313],
        [-0.1697,  0.1650],
        [-0.2202,  0.1801]], grad_fn=<SliceBackward0>)
Trainable parameters: 524674


## 04. Training MLP model

### 4.1 Configure Loss Function and Optimizer

In [14]:
# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer - update params
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001)


### 4.2 Perform 1 training step

Forward pass (model makes prediction) > Calculate loss (compare predictions against actual labels) > Backward pass (calculate gradients) > Optimizer step (update model params)

In [25]:
# Set model to training mode
model.train()

# Get 1 batch
images, labels = next(iter(train_loader))
print("Images:", images.shape)
print("Labels:", labels.shape)

# Clear previous gradients
optimizer.zero_grad()

# Make predictions
outputs = model(images)
print("Predictions:", outputs.shape)

# Calculate Loss
loss = criterion(outputs, labels)
print("Loss:", loss.item())

# Backpropagation & parameter update(optimizer)
loss.backward()
optimizer.step()

Images: torch.Size([32, 1, 64, 64])
Labels: torch.Size([32])
Predictions: torch.Size([32, 2])
Loss: 0.2865062952041626


### 4.3 Train the MLP for 1 epoch (69 batches of 32 images)

In [27]:
model.train()
running_loss = 0.0

# Iterate through all 69 batches from DataLoader
for images, labels, in train_loader:
    # 1. Clear previous gradients
    optimizer.zero_grad()
    # 2. Forward pass
    outputs = model(images)
    # 3. Calculate loss
    loss = criterion(outputs, labels)
    # 4. Backpropagation
    loss.backward()
    # 5. Update model parameters
    optimizer.step()
    # 6. Accumulate loss
    running_loss += loss.item()

# Average loss across all batches in that epoch
epoch_loss = running_loss / len(train_loader)
print(f"Training loss: {epoch_loss:.4f}")

Training loss: 0.3687


### 4.4 Validate the MLP

Validation dataset containing 554 images

In [28]:
# Switch model to evaluation mode
model.eval()

val_loss = 0.0

# torch.no_grad() tells pytorch not to calculate gradients (not needed as not updating model, hence no optimizer.step() too, as validation data must not update model's weight)
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
epoch_val_loss = val_loss / len(val_loader)
print(f"Validation loss: {epoch_val_loss:.4f}")        

Validation loss: 0.3673


### 4.5 Train for multiple epochs

Train and validate across 10 epochs

In [ ]:
NUM_EPOCHS = 10
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
    epoch_val_loss = running_val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    # Results
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )            

Epoch [1/10]Train Loss: 0.3620Val Loss: 0.3562
Epoch [2/10]Train Loss: 0.3436Val Loss: 0.3525
Epoch [3/10]Train Loss: 0.3375Val Loss: 0.3749
Epoch [4/10]Train Loss: 0.3547Val Loss: 0.3695
Epoch [5/10]Train Loss: 0.3481Val Loss: 0.3712
Epoch [6/10]Train Loss: 0.3368Val Loss: 0.3967
Epoch [7/10]Train Loss: 0.3408Val Loss: 0.3523
Epoch [8/10]Train Loss: 0.3271Val Loss: 0.3486
Epoch [9/10]Train Loss: 0.3294Val Loss: 0.3504
Epoch [10/10]Train Loss: 0.3260Val Loss: 0.3555


## 05. Evaluate MLP Model

Evaluate validation predictions

In [32]:
# Switch model to evaluation mode
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        # Select class with higher logit
        predictions = torch.argmax(outputs, dim=1)
        all_predictions.extend(predictions.tolist())
        all_labels.extend(labels.tolist())

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_predictions, labels=[0, 1])
print(cm)

# Evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, zero_division=0)
recall = recall_score(all_labels, all_predictions, zero_division=0)
f1 = f1_score(all_labels, all_predictions, zero_division=0)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 score: {f1:.4f}")

[[474  12]
 [ 64   4]]
Accuracy: 0.8628
Precision: 0.2500
Recall: 0.0588
F1 score: 0.0952


This model predicted 474 TNs, 4 TPs, 64 FNs and 12 FPs. For a medical imaging model, the 64 FNs are of biggest concern, as the model missed 64 out of 68 actual effusion images. This shows in the low recall score of 5.88%, which indicates how many effusion cases were detected out of all the actual effusion cases. 

This is likely the effect of class imbalance, as the model learnt to favour "No Finding" too strongly. 

In the next section, we will attempt to handle class imbalance. 

## 06. Handling Class Imbalance

### 6.1 Introduce Class Weights

In [ ]:
class_counts = torch.bincount(y_train)
class_weights = len(y_train) / (2 * class_counts)
print("Class counts:", class_counts)
print("Class weights:", class_weights)
# Effusion receives much higher weight due to it being minority class

Class counts: tensor([1939,  269])
Class weights: tensor([0.5694, 4.1041])


### 6.2 Train Weighted MLP

In [ ]:
# Start with fresh model so it doesn't retain what it learned during baseline training
model = ChestXrayMLP()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001)

# Update loss function to penalise incorrect effusion predictions more heavily
critertion = nn.CrossEntropyLoss(weight=class_weights)

In [35]:
# Retrain new weighted model using same 10-epoch training and validation loop
NUM_EPOCHS = 10
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
    epoch_val_loss = running_val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    # Results
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )            

Epoch [1/10] Train Loss: 0.3410 Val Loss: 0.3560
Epoch [2/10] Train Loss: 0.3275 Val Loss: 0.3418
Epoch [3/10] Train Loss: 0.3272 Val Loss: 0.3482
Epoch [4/10] Train Loss: 0.3256 Val Loss: 0.3424
Epoch [5/10] Train Loss: 0.3253 Val Loss: 0.3443
Epoch [6/10] Train Loss: 0.3211 Val Loss: 0.3386
Epoch [7/10] Train Loss: 0.3175 Val Loss: 0.3510
Epoch [8/10] Train Loss: 0.3190 Val Loss: 0.4154
Epoch [9/10] Train Loss: 0.3350 Val Loss: 0.3599
Epoch [10/10] Train Loss: 0.3217 Val Loss: 0.3560


### 6.3 Evaluate Weighted MLP

In [36]:
# Switch model to evaluation mode
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        # Select class with higher logit
        predictions = torch.argmax(outputs, dim=1)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_predictions)
print(cm)

# Evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, zero_division=0)
recall = recall_score(all_labels, all_predictions, zero_division=0)
f1 = f1_score(all_labels, all_predictions, zero_division=0)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 score: {f1:.4f}")

[[478   8]
 [ 67   1]]
Accuracy: 0.8646
Precision: 0.1111
Recall: 0.0147
F1 score: 0.0260


Native PyTorch training loop

1. Set model to training mode
model.train()  
2. Initialise var to accumulate training loss
train_loss = 0.0
3. Loop through training data
for images, labels in train_loader:
    4. Clear gradients
    optimizer.zero_grad()
    5. Forward pass: store model's logits(predictions) for current batch
    outputs = model(images)
    6. Calculate loss
    loss = criterion(outputs, labels)
    7. Backpropagation
    loss.backward()  
    8. Use gradients to update model's weights
    optimizer.step() 
    9. Accumulate training loss
    train_loss += loss.item()
10. Calculate average epoch loss
epoch_loss = train_loss / len(train_loader)

Before that, we should have 
1. Prepared data
train_loader, val_loader
2. Defined model
model = ChestXrayMLP()
3. Initialise loss function (ONCE, before training)
criterion - nn.CrossEntropyLoss()
4. Initialise optimizer (ONCE, before training)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
5. Start training loop (every batch)
> model.train() is called at the beginning of each training epoch

## 07. Conclusion

The baseline MLP performed poorly in detecting effusion in chest x-ray images. A possible reason for the poor results could be due to class imbalance, which can make accuracy misleading. The model appeared to learn the majority (no finding) class while performing poorly on the minority (effusion) class. 

A weighted MLP was trained to balance the classes more. While the losses appeared to be an improvement compared to the baseline MLP model, evaluation requires more than looking at just the loss. Other metrics like accuracy, precision, recall, and f1-score were calculated, and it showed that the weighted MLP did not perform better than the baseline MLP. 

As the purpose of this exercise was to understand the mechanics of a neural network, not to produce a clinically useful classifier, hence this notebook will be concluded at this point. We will revisit the model's shortcomings when we return after learning CNNs and more appropriate image-classification techniques.